# Controlled-Z: phase, basis changes, and entanglement

By the end, you will derive the controlled-Z matrix, distinguish a phase change from a probability change, and verify the difference between a one-sided and a two-sided Hadamard basis change. You need basic matrix multiplication and the Born rule. The expected output includes exact operator checks, a reduced-state purity, and seeded measurement frequencies.

**Prerequisites:** Python 3.12 or later and the tested environment in the [collection README](../../../README.md). Install the [root requirements](../../../requirements.txt) into an isolated environment before opening this notebook; select that environment's kernel. Run all cells from a fresh kernel in order. This notebook makes no network calls, reads no external files, and writes no artifacts by default.

**Reproducibility:** all examples and data here are synthetic. Simulator and sampling seeds are explicit. Assertions distinguish mathematical identities, finite-shot checks, and integration contracts. Approximate comparisons state their tolerance.


Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../../AI_DISCLOSURE.md), [license](../../../LICENSE), and [attribution](../../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Fix the basis before multiplying matrices

Qiskit labels two-qubit basis vectors as $|q_1q_0\rangle$: integer index $b$ stores qubit $j$ in bit $(b\gg j)\&1$. We use that convention everywhere. The gate is
$CZ=\operatorname{diag}(1,1,1,-1)$, so only $|11\rangle$ receives a minus sign. Since $CZ^2=I$, it is both unitary and Hermitian.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator, Statevector, partial_trace
from qiskit_aer import AerSimulator


def show_plot():
    import io

    from IPython.display import Image, display

    buffer = io.BytesIO()
    plt.gcf().savefig(buffer, format="png", dpi=120, bbox_inches="tight")
    display(Image(data=buffer.getvalue()))
    plt.close()


H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
CZ = np.diag([1, 1, 1, -1]).astype(complex)
H2 = np.kron(H, H)
assert np.allclose(CZ.conj().T @ CZ, np.eye(4), atol=1e-12)
assert np.allclose(CZ @ CZ, np.eye(4), atol=1e-12)
pd.DataFrame(CZ.real, index=["00", "01", "10", "11"], columns=["00", "01", "10", "11"])

## 2. A relative phase is invisible in one measurement basis

Starting in $|++\rangle$, all four computational-basis probabilities stay $1/4$ after CZ. The state is nevertheless entangled. For a bipartite pure state, a mixed reduced state certifies entanglement. Here tracing out either qubit gives $I/2$, with purity $\mathrm{Tr}(\rho^2)=1/2$. A single computational-basis histogram cannot distinguish this state from the unentangled input.


In [ ]:
prepare = QuantumCircuit(2)
prepare.h([0, 1])
before = Statevector.from_instruction(prepare)
prepare.cz(0, 1)
after = Statevector.from_instruction(prepare)
assert np.allclose(before.probabilities(), after.probabilities(), atol=1e-12)
rho0 = partial_trace(after, [1]).data
purity = float(np.trace(rho0 @ rho0).real)
assert np.allclose(rho0, np.eye(2) / 2, atol=1e-12)
assert abs(purity - 0.5) < 1e-12
print("Probabilities:", after.probabilities(), "reduced purity:", purity)

## 3. Which Hadamards produce CNOT?

Conjugating CZ by H on **only its target** yields CX. With control $q_0$ and target $q_1$, the change-of-basis matrix is $H\otimes I$. Applying H on both qubits instead gives $U=(H\otimes H)CZ(H\otimes H)$, a different operator. Compare full matrices, not just their action on $|00\rangle$.

In the matrix plot, rows and columns use the same discrete `q1q0` basis labels as the derivation. Color encodes a real matrix entry, not a measurement probability.


In [ ]:
cx = QuantumCircuit(2)
cx.cx(0, 1)
H_target = np.kron(H, np.eye(2))
assert np.allclose(H_target @ CZ @ H_target, Operator(cx).data, atol=1e-12)
U = H2 @ CZ @ H2
sandwich = QuantumCircuit(2)
sandwich.h([0, 1])
sandwich.cz(0, 1)
sandwich.h([0, 1])
assert np.allclose(Operator(sandwich).data, U, atol=1e-12)
assert not np.allclose(U, Operator(cx).data)
assert np.allclose(U @ U, np.eye(4), atol=1e-12)
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(U.real, vmin=-1, vmax=1, cmap="RdBu_r")
basis_labels = [f"{index:02b}" for index in range(4)]
ax.set_xticks(range(4), labels=basis_labels)
ax.set_yticks(range(4), labels=basis_labels)
ax.set(xlabel="Input basis (q1q0)", ylabel="Output basis (q1q0)", title="H⊗H · CZ · H⊗H")
fig.colorbar(image, ax=ax, label="Real matrix entry")
show_plot()

## 4. Check sampling without mistaking it for exact algebra

We use 8,192 shots and seed 17. The empirical probabilities should lie within 0.04 of the exact values. Hoeffding's bound and a union bound give failure probability at most $8\exp(-2N\epsilon^2)$ for these four bins; this is a sampling check, not a hardware error budget.


In [ ]:
shots, tolerance = 8192, 0.04
measured = sandwich.copy()
measured.measure_all()
sim = AerSimulator()
counts = (
    sim.run(transpile(measured, sim, seed_transpiler=17), shots=shots, seed_simulator=17)
    .result()
    .get_counts()
)
observed = np.array([counts.get(f"{i:02b}", 0) / shots for i in range(4)])
expected = Statevector.from_instruction(sandwich).probabilities()
assert np.max(np.abs(observed - expected)) < tolerance
print(
    pd.DataFrame({"basis": [f"{i:02b}" for i in range(4)], "exact": expected, "sampled": observed})
)
print("Union-bound failure probability:", 8 * np.exp(-2 * shots * tolerance**2))

## Limits, cleanup, and next steps

The simulator has no gate or readout errors. A uniform histogram does not prove entanglement; the reduced-state calculation uses simulator state access unavailable on hardware. On hardware, estimate complementary observables or perform suitable tomography. Change the input phase and predict the matrix result before sampling again.

No files, cloud jobs, or credentials were created. Close figure windows or restart the kernel to release simulator memory.

Primary references: [IBM Qiskit bit ordering](https://quantum.cloud.ibm.com/docs/en/guides/bit-ordering), [CZGate API](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.CZGate), [Statevector API](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Statevector).
